# 逆翻訳で擬似対訳データを作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/26_machine_translation/back_translation.ipynb)

NLP クックブック 26-11 ／ 機械翻訳

対訳データは少ないが、訳文側の言語だけのテキスト（単言語データ）は大量にある。これを使って翻訳モデルの学習データを増やしたい。

In [1]:
!pip install -q datasets sentencepiece sacremoses sacrebleu

In [2]:
import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

# 英日翻訳モデルを強化したい場合: 日本語の単言語データを「日 → 英」のモデルで英語に訳し、(機械訳の英語, 人が書いた日本語) の組を作る
name = "Helsinki-NLP/opus-mt-ja-en"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForSeq2SeqLM.from_pretrained(name).eval()

@torch.no_grad()
def translate(sentences, batch_size=32, **kwargs):
    out = []
    for i in range(0, len(sentences), batch_size):
        enc = tokenizer(sentences[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=128)
        out += tokenizer.batch_decode(model.generate(**enc, max_new_tokens=128, **kwargs), skip_special_tokens=True)
    return out

mono = ["吾輩は猫である。名前はまだ無い。", "国境の長いトンネルを抜けると雪国であった。", "今日は朝から雨が降っていて、洗濯物が乾かない。",
        "新しいスマートフォンを買ったが、まだ使い方がよく分からない。", "この店のラーメンは、スープが濃厚でとてもおいしい。",
        "明日の会議は午後 3 時に変更になりました。", "彼女は大学で自然言語処理を研究している。", "駅前に大きな書店ができたので、帰りに寄ってみた。"]
pseudo_src = translate(mono, num_beams=4)
for en, ja in zip(pseudo_src, mono):
    print(f"{en}\n    ← {ja}")

I'm a cat. I haven't got a name yet.
    ← 吾輩は猫である。名前はまだ無い。
Through a long tunnel on the border, it was a snow country.
    ← 国境の長いトンネルを抜けると雪国であった。
It has been raining since this morning, so the laundry isn't dry.
    ← 今日は朝から雨が降っていて、洗濯物が乾かない。
I bought a new smart phone, but I still don't know how to use it.
    ← 新しいスマートフォンを買ったが、まだ使い方がよく分からない。
The ramen of this store is rich in soup and very delicious.
    ← この店のラーメンは、スープが濃厚でとてもおいしい。
The meeting tomorrow changed at 3 p.m.
    ← 明日の会議は午後 3 時に変更になりました。
She studies natural language processing at college.
    ← 彼女は大学で自然言語処理を研究している。
A large bookstore was built in front of the station, so I stopped by on my way home.
    ← 駅前に大きな書店ができたので、帰りに寄ってみた。


In [3]:
# 逆翻訳の作り方を変える: ビームサーチ (最も確からしい訳) / サンプリング (多様だが雑音が多い)
torch.manual_seed(0)
for ja in mono[2:5]:
    print(ja)
    print("   ビーム    :", translate([ja], num_beams=4)[0])
    for _ in range(3):
        print("   サンプリング:", translate([ja], do_sample=True, top_k=0, temperature=1.0)[0])

今日は朝から雨が降っていて、洗濯物が乾かない。
   ビーム    : It has been raining since this morning, so the laundry isn't dry.
   サンプリング: It has been raining since this morning, so the laundry isn't dry.
   サンプリング: It has been raining since this morning, so the laundry won't dry.
   サンプリング: It has been raining since this morning, so the laundry won't dry.
新しいスマートフォンを買ったが、まだ使い方がよく分からない。
   ビーム    : I bought a new smart phone, but I still don't know how to use it.
   サンプリング: I bought a new smart phone, but I still don't know how to use it.
   サンプリング: I bought a new smart phone, but I still don't know how to use it.
   サンプリング: I bought a new smart phone, but I still don't know how to use it.
この店のラーメンは、スープが濃厚でとてもおいしい。
   ビーム    : The ramen of this store is rich in soup and very delicious.
   サンプリング: The Rammen of this shop are rich and very delicious in soup.
   サンプリング: The ramen of this store is rich in soup and very delicious.
   サンプリング: The ramen of this store is rich in soup and very delicious.


In [4]:
# まとまった量を作って保存する。擬似データであることが分かる印 (タグ) を原言語側の先頭に付けておく
from datasets import load_dataset

ds = load_dataset("Helsinki-NLP/opus-100", "en-ja", split="test")
mono_ja = [ex["translation"]["ja"] for ex in ds.select(range(200)) if 5 <= len(ex["translation"]["ja"]) <= 60]
bt_en = translate(mono_ja, num_beams=4)
with open("backtranslated.tsv", "w", encoding="utf-8") as f:
    for en, ja in zip(bt_en, mono_ja):
        f.write(f"<BT> {en}\t{ja}\n")
print(len(mono_ja), "文対を保存")
!head -n 5 backtranslated.tsv

# 逆翻訳の質の目安: このデータにはたまたま人手の英訳があるので、それと比べる
import sacrebleu
refs = [ex["translation"]["en"] for ex in ds.select(range(200)) if 5 <= len(ex["translation"]["ja"]) <= 60]
print(sacrebleu.corpus_bleu(bt_en, [refs]))

190 文対を保存
<BT> If that happens,	もしそうなったら
<BT> I'm sorry, but there's a wall.	お父さんか？ 悪いけど、壁があるんだけど
<BT> Hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey, hey.	フリスビー　知ってる？
<BT> In my office.	オフィスへ
<BT> The farmer sows corn in spring.	農夫は春にとうもろこしの種をまく。
BLEU = 17.59 48.8/25.3/16.3/11.4 (BP = 0.805 ratio = 0.821 hyp_len = 1683 ref_len = 2049)


- 逆翻訳 (back-translation) は、訳文側が「人の書いた自然な文」で、原文側が「機械訳」の対訳データを作る。訳文側の質が保たれるので、原文側に多少の誤訳があっても、翻訳モデルの流暢さが上がる。
- 作った擬似対訳は、本物の対訳データと混ぜて学習に使う。比率は 1:1 から始めて、検証データで調整するのが一般的。
- 擬似データの先頭に `<BT>` のようなタグを付けておくと、モデルが本物と擬似データを区別して学習できる (tagged back-translation)。
- データが大量にあるときは、ビームサーチよりサンプリングで作った逆翻訳のほうが効果が高いと報告されている。原文側が多様になり、学習の信号が増えるため。データが少ないときは、ビームサーチのほうが安全。
- 逆翻訳に使うモデル自体の質が低いと、効果は小さい。次項のフィルタリングで質の低い組を捨てる方法がある。